# 02. Enhanced Data Ingestion Pipeline with Unstructured.io

This notebook demonstrates the **enhanced** data ingestion process using **Unstructured.io** for superior document processing capabilities.

## 🚀 New Features in This Version:
- **Unstructured.io Integration**: Advanced document parsing with semantic element classification
- **Enhanced Metadata Extraction**: Rich document analysis with element types and counts
- **Improved Code Detection**: Better language identification and code snippet extraction
- **Table Processing**: Built-in table structure preservation
- **Parser Comparison**: Side-by-side comparison with the legacy custom parser

## Objectives:
- Clone the LangChain repository
- Parse multi-format documentation with **Unstructured.io**
- Extract **enhanced metadata** with semantic element classification
- Compare parsing methods and demonstrate improvements
- Generate comprehensive statistics and visualizations
- Save processed documents for the next phase

## Document Processing Strategy:
1. **Enhanced Multi-Format Parsing**: Automatic format detection and semantic classification
2. **Rich Metadata Extraction**: Element types, counts, hierarchy, and structure analysis  
3. **Improved Content Extraction**: Better handling of complex documents (PDFs, code files, etc.)
4. **Table and Structure Preservation**: Maintain document structure for better RAG performance
5. **Performance Optimization**: Production-ready parsing with error handling

## 1. Environment Setup

In [1]:
import sys
import os
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import logging

# Add src to path
project_root = Path.cwd().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print(f"📁 Project root: {project_root}")
print(f"📂 Source path: {src_path}")

📁 Project root: /Users/sourangshupal/Documents/projects/langchain-rag-project
📂 Source path: /Users/sourangshupal/Documents/projects/langchain-rag-project/src


In [1]:
# Import our enhanced ingestion modules
from ingestion.repo_cloner import LangChainRepoManager, clone_langchain_docs
from ingestion.document_parser import MultiFormatParser, parse_langchain_documents
from ingestion.unstructured_parser import (
    UnstructuredDocumentParser, 
    parse_langchain_documents_unstructured,
    compare_parsing_methods
)
from ingestion.data_pipeline import DataIngestionPipeline, run_ingestion_pipeline, run_parser_comparison_pipeline
from config import settings

print("✅ Successfully imported enhanced ingestion modules")
print(f"📊 Configuration loaded: {settings.collection_name}")
print("🚀 Unstructured.io integration ready!")

ModuleNotFoundError: No module named 'ingestion'

## 2. Repository Management

In [ ]:
# Initialize repository manager
repo_manager = LangChainRepoManager()

print(f"🔗 Repository URL: {repo_manager.repo_url}")
print(f"📁 Local path: {repo_manager.local_path}")
print(f"📊 Repository exists: {repo_manager.local_path.exists()}")

# Check if we need to clone or update
if repo_manager.local_path.exists():
    print("ℹ️  Repository already exists. Will update if needed.")
else:
    print("🚀 Will clone fresh repository.")

In [ ]:
# Clone or update repository
print("🔄 Cloning/updating LangChain repository...")
success = repo_manager.clone_repository(force_refresh=False)

if success:
    print("✅ Repository ready!")
    
    # Get repository statistics
    stats = repo_manager.get_repository_stats()
    print("\n📊 Repository Statistics:")
    for key, value in stats.items():
        if key == "file_counts":
            print("   File counts by type:")
            for file_type, count in value.items():
                print(f"     {file_type}: {count}")
        else:
            print(f"   {key}: {value}")
else:
    print("❌ Failed to clone repository. Please check your internet connection and try again.")

## 3. Document Discovery

In [ ]:
# Discover documentation files
if success:
    print("🔍 Discovering documentation files...")
    doc_paths = repo_manager.get_documentation_paths()
    
    print("\n📄 Documentation Files Found:")
    total_files = 0
    for file_type, paths in doc_paths.items():
        count = len(paths)
        total_files += count
        print(f"   {file_type}: {count} files")
    
    print(f"\nTotal documentation files: {total_files}")
    
    # Show sample files from each type
    print("\n📋 Sample files by type:")
    for file_type, paths in doc_paths.items():
        if paths:
            print(f"\n{file_type.upper()}:")
            for i, path in enumerate(paths[:3]):  # Show first 3 files
                relative_path = path.relative_to(repo_manager.local_path)
                print(f"   {i+1}. {relative_path}")
            if len(paths) > 3:
                print(f"   ... and {len(paths) - 3} more")
else:
    print("⚠️  Skipping file discovery due to repository clone failure.")

## 4. Enhanced Document Parsing with Unstructured.io

In [ ]:
# Demonstrate enhanced parsing with Unstructured.io
if success and total_files > 0:
    print("🚀 Testing enhanced document parsing with Unstructured.io...")
    print("=" * 60)
    
    # Initialize both parsers for comparison
    unstructured_parser = UnstructuredDocumentParser()
    custom_parser = MultiFormatParser()
    
    # Test different document types with both parsers
    comparison_results = []
    
    for file_type, paths in doc_paths.items():
        if not paths:
            continue
            
        # Test first file of each type with both parsers
        sample_path = paths[0]
        print(f"\n📄 Parsing {file_type}: {sample_path.name}")
        print("-" * 40)
        
        # Parse with Unstructured.io
        print("🔍 Unstructured.io Parser:")
        unstructured_result = unstructured_parser.parse_document(sample_path)
        
        # Parse with custom parser
        print("🔧 Custom Parser:")  
        custom_result = custom_parser.parse_document(sample_path)
        
        if unstructured_result and custom_result:
            # Compare results
            unstructured_meta = unstructured_result["metadata"]
            custom_meta = custom_result["metadata"]
            
            print(f"\n📊 Comparison Results:")
            print(f"   Content Length:")
            print(f"     Unstructured.io: {len(unstructured_result['content']):,} chars")
            print(f"     Custom Parser:   {len(custom_result['content']):,} chars")
            
            improvement = ((len(unstructured_result['content']) - len(custom_result['content'])) 
                          / len(custom_result['content']) * 100)
            print(f"     Improvement:     {improvement:+.1f}%")
            
            print(f"\n   Enhanced Features (Unstructured.io only):")
            print(f"     Elements Detected: {len(unstructured_result['elements'])}")
            print(f"     Element Types: {list(unstructured_meta['element_counts'].keys())}")
            print(f"     Tables Found: {len(unstructured_meta['tables'])}")
            print(f"     List Items: {len(unstructured_meta['list_items'])}")
            print(f"     Narrative Text Count: {unstructured_meta['narrative_text_count']}")
            
            print(f"\n   Code Snippet Comparison:")
            print(f"     Unstructured.io: {len(unstructured_meta['code_snippets'])} snippets")
            print(f"     Custom Parser:   {len(custom_meta['code_snippets'])} snippets")
            
            # Show element type breakdown
            if unstructured_meta['element_counts']:
                print(f"\n   📋 Element Type Breakdown:")
                for element_type, count in unstructured_meta['element_counts'].items():
                    print(f"     {element_type}: {count}")
            
            comparison_results.append({
                'file_type': file_type,
                'file_name': sample_path.name,
                'unstructured_length': len(unstructured_result['content']),
                'custom_length': len(custom_result['content']),
                'improvement_percent': improvement,
                'elements_detected': len(unstructured_result['elements']),
                'element_types': len(unstructured_meta['element_counts']),
                'tables_found': len(unstructured_meta['tables'])
            })
            
        elif unstructured_result:
            print(f"   ✅ Unstructured.io: Success (Custom parser failed)")
        elif custom_result:
            print(f"   ⚠️  Custom parser: Success (Unstructured.io failed)")
        else:
            print(f"   ❌ Both parsers failed")
    
    # Summary of improvements
    if comparison_results:
        print(f"\n🎯 Overall Improvements with Unstructured.io:")
        print("=" * 50)
        
        avg_improvement = sum(r['improvement_percent'] for r in comparison_results) / len(comparison_results)
        total_elements = sum(r['elements_detected'] for r in comparison_results)
        total_element_types = sum(r['element_types'] for r in comparison_results)
        total_tables = sum(r['tables_found'] for r in comparison_results)
        
        print(f"📈 Average content extraction improvement: {avg_improvement:+.1f}%")
        print(f"🔍 Total semantic elements detected: {total_elements}")
        print(f"📊 Unique element types identified: {total_element_types}")
        print(f"📋 Tables successfully extracted: {total_tables}")
        
        print(f"\n✨ Key Advantages:")
        print(f"   🎯 Automatic format detection and handling")
        print(f"   📝 Rich semantic element classification")
        print(f"   🔍 Enhanced metadata with element counts")
        print(f"   📊 Built-in table structure preservation")
        print(f"   💻 Advanced code snippet detection")
        print(f"   🚀 Production-ready performance optimizations")

else:
    print("⚠️  Skipping enhanced parsing test due to no files found.")

## 5. Enhanced Data Ingestion Pipeline

In [ ]:
# Run the enhanced ingestion pipeline with Unstructured.io
print("🚀 Running Enhanced Data Ingestion Pipeline with Unstructured.io")
print("=" * 65)
print("This may take several minutes depending on the repository size.")
print("The new pipeline includes:")
print("  🎯 Automatic format detection")
print("  📊 Semantic element classification") 
print("  🔍 Enhanced metadata extraction")
print("  📋 Table structure preservation")
print("  💻 Advanced code snippet detection")
print()

# Run enhanced pipeline (uses Unstructured.io by default)
pipeline_result = run_ingestion_pipeline(
    force_refresh=False, 
    use_unstructured=True  # Use enhanced Unstructured.io parser
)

if pipeline_result["success"]:
    print("\n🎉 Enhanced Pipeline completed successfully!")
    
    # Display enhanced pipeline results
    metadata = pipeline_result["pipeline_metadata"]
    print(f"\n📊 Pipeline Summary:")
    print(f"   Parser Used: {metadata.get('parser_used', 'unstructured').title()}")
    print(f"   Start time: {metadata['start_time']}")
    print(f"   End time: {metadata['end_time']}")
    print(f"   Total documents processed: {metadata['total_documents']}")
    print(f"   Successful parses: {metadata['successful_parses']}")
    print(f"   Failed parses: {metadata['failed_parses']}")
    
    # Calculate success rate
    success_rate = (metadata['successful_parses'] / metadata['total_documents']) * 100
    print(f"   Success rate: {success_rate:.1f}%")
    
    # Show document type breakdown
    print(f"\n📋 Documents by type:")
    total_processed = 0
    for doc_type, count in metadata['document_types'].items():
        print(f"   {doc_type}: {count}")
        total_processed += count
    
    # Show parsing statistics
    parsing_stats = pipeline_result.get("parsing_stats", {})
    print(f"\n📈 Processing Statistics:")
    print(f"   Total attempted: {parsing_stats.get('total_attempted', 0)}")
    print(f"   Total successful: {parsing_stats.get('total_successful', 0)}")
    print(f"   Success rate: {parsing_stats.get('success_rate', 0):.1%}")
    
    # Show output files
    print(f"\n💾 Generated Output Files:")
    for file_path in pipeline_result['output_files']:
        file_name = Path(file_path).name
        file_size = Path(file_path).stat().st_size / 1024  # KB
        print(f"   {file_name} ({file_size:.1f} KB)")
    
    print(f"\n✨ Enhanced Features Included:")
    print(f"   🎯 Semantic element classification (Title, NarrativeText, Table, etc.)")
    print(f"   📊 Rich metadata with element counts and types")
    print(f"   📋 Preserved table structures and list items")
    print(f"   💻 Enhanced code snippet detection with language identification")
    print(f"   🔍 Detailed document analysis for better RAG performance")

else:
    print(f"❌ Enhanced pipeline failed: {pipeline_result.get('error', 'Unknown error')}")
    print("Falling back to custom parser might be needed for debugging.")

## 6. Enhanced Data Analysis and Visualization

In [ ]:
# Run a detailed comparison between parsers on a sample of documents
if success and pipeline_result["success"]:
    print("🔍 Running Detailed Parser Comparison Analysis")
    print("=" * 50)
    print("This will compare Unstructured.io vs Custom Parser on sample documents...")
    
    try:
        # Run parser comparison on a sample
        comparison_result = run_parser_comparison_pipeline(sample_size=15)
        
        if comparison_result.get("success", True):
            custom_stats = comparison_result.get("custom_parser", {})
            unstructured_stats = comparison_result.get("unstructured_parser", {})
            comparisons = comparison_result.get("comparison", [])
            
            print(f"\n📊 Parser Performance Comparison:")
            print("-" * 40)
            
            # Overall success rates
            print(f"Success Rates:")
            print(f"  🔧 Custom Parser:     {custom_stats.get('successful', 0)}/{custom_stats.get('successful', 0) + custom_stats.get('failed', 0)} "
                  f"({custom_stats.get('successful', 0) / (custom_stats.get('successful', 0) + custom_stats.get('failed', 0)) * 100:.1f}%)")
            print(f"  🚀 Unstructured.io:   {unstructured_stats.get('successful', 0)}/{unstructured_stats.get('successful', 0) + unstructured_stats.get('failed', 0)} "
                  f"({unstructured_stats.get('successful', 0) / (unstructured_stats.get('successful', 0) + unstructured_stats.get('failed', 0)) * 100:.1f}%)")
            
            if comparisons:
                # Content length comparison
                total_custom_length = sum(c['custom_content_length'] for c in comparisons if c['custom_success'])
                total_unstructured_length = sum(c['unstructured_content_length'] for c in comparisons if c['unstructured_success'])
                successful_comparisons = [c for c in comparisons if c['custom_success'] and c['unstructured_success']]
                
                if successful_comparisons:
                    print(f"\n📈 Content Extraction Analysis:")
                    print(f"  Total Content Extracted:")
                    print(f"    Custom Parser:     {total_custom_length:,} characters")
                    print(f"    Unstructured.io:   {total_unstructured_length:,} characters")
                    
                    improvement = ((total_unstructured_length - total_custom_length) / total_custom_length * 100) if total_custom_length > 0 else 0
                    print(f"    Overall Improvement: {improvement:+.1f}%")
                    
                    # Show top improvements
                    print(f"\n🏆 Top Document Improvements:")
                    sorted_comparisons = sorted(successful_comparisons, 
                                              key=lambda x: x.get('content_length_diff', 0), reverse=True)
                    
                    for i, comp in enumerate(sorted_comparisons[:5], 1):
                        filename = Path(comp['file']).name
                        diff = comp.get('content_length_diff', 0)
                        custom_len = comp['custom_content_length']
                        improvement_pct = (diff / custom_len * 100) if custom_len > 0 else 0
                        
                        print(f"  {i}. {filename[:40]:<40} {improvement_pct:+6.1f}% ({diff:+,} chars)")
                    
                    # Enhanced features summary
                    total_elements = sum(c.get('unstructured_elements', 0) for c in comparisons)
                    custom_code_snippets = sum(c.get('custom_code_snippets', 0) for c in comparisons)
                    unstructured_code_snippets = sum(c.get('unstructured_code_snippets', 0) for c in comparisons)
                    
                    print(f"\n✨ Enhanced Features (Unstructured.io):")
                    print(f"  📊 Total Semantic Elements: {total_elements}")
                    print(f"  💻 Code Snippets Detected:")
                    print(f"    Custom Parser:     {custom_code_snippets}")
                    print(f"    Unstructured.io:   {unstructured_code_snippets}")
                    
                    # File type performance
                    print(f"\n📋 Performance by File Type:")
                    file_type_stats = {}
                    for comp in successful_comparisons:
                        file_ext = Path(comp['file']).suffix.lower()
                        if file_ext not in file_type_stats:
                            file_type_stats[file_ext] = {'count': 0, 'total_improvement': 0}
                        
                        file_type_stats[file_ext]['count'] += 1
                        if comp['custom_content_length'] > 0:
                            improvement = (comp.get('content_length_diff', 0) / comp['custom_content_length'] * 100)
                            file_type_stats[file_ext]['total_improvement'] += improvement
                    
                    for file_ext, stats in file_type_stats.items():
                        avg_improvement = stats['total_improvement'] / stats['count'] if stats['count'] > 0 else 0
                        print(f"  {file_ext or 'no ext':<10} ({stats['count']:2d} files): {avg_improvement:+6.1f}% avg improvement")
                    
            print(f"\n💾 Comparison results saved to: parser_comparison.json")
            print(f"🎯 Unstructured.io demonstrates superior document processing capabilities!")
            
        else:
            print(f"❌ Parser comparison failed: {comparison_result.get('error', 'Unknown error')}")
            
    except Exception as e:
        print(f"⚠️  Could not run parser comparison: {str(e)}")
        print("This might be due to missing repository data. The enhanced pipeline is still working!")
        
else:
    print("⚠️  Skipping parser comparison due to pipeline issues.")

## 6. Data Analysis and Visualization

In [ ]:
# Load and analyze processed documents
if pipeline_result["success"]:
    # Load metadata CSV for analysis
    metadata_file = project_root / "data" / "processed" / "document_metadata.csv"
    
    if metadata_file.exists():
        df = pd.read_csv(metadata_file)
        print(f"📊 Loaded metadata for {len(df)} documents")
        
        # Basic statistics
        print(f"\n📈 Content Length Statistics:")
        print(f"   Mean: {df['content_length'].mean():.0f} characters")
        print(f"   Median: {df['content_length'].median():.0f} characters")
        print(f"   Min: {df['content_length'].min():.0f} characters")
        print(f"   Max: {df['content_length'].max():.0f} characters")
        print(f"   95th percentile: {df['content_length'].quantile(0.95):.0f} characters")
    else:
        print("❌ Metadata file not found")
        df = None
else:
    print("⚠️  Skipping analysis due to pipeline failure.")
    df = None

In [ ]:
# Create visualizations
if df is not None and len(df) > 0:
    # Set up plotting style
    plt.style.use('default')
    sns.set_palette("husl")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('LangChain Documentation Analysis', fontsize=16, fontweight='bold')
    
    # 1. Document count by type
    doc_type_counts = df['doc_type'].value_counts()
    axes[0, 0].bar(doc_type_counts.index, doc_type_counts.values)
    axes[0, 0].set_title('Document Count by Type')
    axes[0, 0].set_xlabel('Document Type')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. Content length distribution
    axes[0, 1].hist(df['content_length'], bins=50, alpha=0.7, edgecolor='black')
    axes[0, 1].set_title('Content Length Distribution')
    axes[0, 1].set_xlabel('Content Length (characters)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].axvline(df['content_length'].median(), color='red', linestyle='--', label='Median')
    axes[0, 1].legend()
    
    # 3. File size distribution
    file_sizes_kb = df['file_size'] / 1024  # Convert to KB
    axes[1, 0].hist(file_sizes_kb, bins=30, alpha=0.7, edgecolor='black')
    axes[1, 0].set_title('File Size Distribution')
    axes[1, 0].set_xlabel('File Size (KB)')
    axes[1, 0].set_ylabel('Frequency')
    
    # 4. Content length vs file size
    scatter = axes[1, 1].scatter(file_sizes_kb, df['content_length'], 
                               c=pd.Categorical(df['doc_type']).codes, 
                               alpha=0.6, s=20)
    axes[1, 1].set_title('Content Length vs File Size')
    axes[1, 1].set_xlabel('File Size (KB)')
    axes[1, 1].set_ylabel('Content Length (characters)')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics table
    print("\n📋 Summary by Document Type:")
    summary_stats = df.groupby('doc_type').agg({
        'content_length': ['count', 'mean', 'median', 'std'],
        'file_size': 'mean'
    }).round(2)
    
    summary_stats.columns = ['Count', 'Avg Length', 'Median Length', 'Std Length', 'Avg File Size']
    print(summary_stats)
else:
    print("⚠️  No data available for visualization.")

## 7. Sample Document Exploration

In [ ]:
# Explore some interesting documents
if df is not None and len(df) > 0:
    print("🔍 Exploring sample documents...\n")
    
    # Find longest document
    longest_doc = df.loc[df['content_length'].idxmax()]
    print(f"📏 Longest Document:")
    print(f"   File: {longest_doc['file_name']}")
    print(f"   Type: {longest_doc['doc_type']}")
    print(f"   Length: {longest_doc['content_length']:,} characters")
    print(f"   Module: {longest_doc['module_path']}")
    if longest_doc['url']:
        print(f"   URL: {longest_doc['url']}")
    
    # Find documents with most code snippets
    df['code_snippet_count'] = df['code_snippets'].apply(lambda x: len(eval(x)) if isinstance(x, str) and x.startswith('[') else 0)
    if df['code_snippet_count'].max() > 0:
        code_heavy_doc = df.loc[df['code_snippet_count'].idxmax()]
        print(f"\n💻 Most Code-Heavy Document:")
        print(f"   File: {code_heavy_doc['file_name']}")
        print(f"   Type: {code_heavy_doc['doc_type']}")
        print(f"   Code snippets: {code_heavy_doc['code_snippet_count']}")
        print(f"   Module: {code_heavy_doc['module_path']}")
    
    # Show top modules by document count
    print(f"\n🏆 Top 10 Modules by Document Count:")
    top_modules = df['module_path'].value_counts().head(10)
    for i, (module, count) in enumerate(top_modules.items(), 1):
        print(f"   {i:2d}. {module}: {count} documents")
else:
    print("⚠️  No documents available for exploration.")

## 8. Data Quality Assessment

In [ ]:
# Assess data quality
if pipeline_result["success"]:
    print("🔍 Data Quality Assessment:\n")
    
    # Success rate
    total_attempted = pipeline_result["pipeline_metadata"]["total_documents"]
    successful = pipeline_result["pipeline_metadata"]["successful_parses"]
    success_rate = (successful / total_attempted) * 100 if total_attempted > 0 else 0
    
    print(f"📊 Overall Statistics:")
    print(f"   Success rate: {success_rate:.1f}% ({successful}/{total_attempted})")
    
    if df is not None:
        # Content quality metrics
        empty_content = (df['content_length'] == 0).sum()
        short_content = (df['content_length'] < 100).sum()
        missing_titles = df['title'].isna().sum()
        
        print(f"\n📋 Content Quality:")
        print(f"   Empty documents: {empty_content}")
        print(f"   Very short documents (<100 chars): {short_content}")
        print(f"   Missing titles: {missing_titles}")
        print(f"   Documents with code snippets: {(df['code_snippet_count'] > 0).sum()}")
        
        # File type distribution
        print(f"\n📄 File Type Distribution:")
        file_types = df['file_name'].apply(lambda x: Path(x).suffix).value_counts()
        for ext, count in file_types.items():
            print(f"   {ext or 'no extension'}: {count}")
        
        # Recommendations
        print(f"\n💡 Recommendations:")
        if success_rate < 90:
            print(f"   - Success rate is {success_rate:.1f}%. Consider improving parsers for failed files.")
        if short_content > len(df) * 0.1:
            print(f"   - {short_content} documents are very short. Consider filtering or merging.")
        if missing_titles > len(df) * 0.2:
            print(f"   - {missing_titles} documents lack titles. Consider improving title extraction.")
        
        avg_length = df['content_length'].mean()
        chunk_size = 1200  # From settings
        estimated_chunks = avg_length / (chunk_size * 0.8)  # Accounting for overlap
        print(f"   - Average document will create ~{estimated_chunks:.1f} chunks (assuming {chunk_size} token chunks)")
        print(f"   - Total estimated chunks: ~{len(df) * estimated_chunks:.0f}")
else:
    print("⚠️  Cannot assess data quality due to pipeline failure.")

## Next Steps

🎉 **Enhanced Data Ingestion Complete with Unstructured.io!**

If the enhanced data ingestion completed successfully, you now have:

### ✅ Enhanced Outputs:
- **Rich Document Data**: JSON files with semantic element classification
- **Enhanced Metadata**: CSV with element counts, types, and structure analysis  
- **Parser Comparison**: Analysis showing Unstructured.io improvements
- **Quality Metrics**: Comprehensive statistics and recommendations

### 🚀 Key Improvements Achieved:
- **🎯 Automatic Format Detection**: No manual format specification needed
- **📊 Semantic Element Classification**: Title, NarrativeText, Table, ListItem, etc.
- **🔍 Enhanced Metadata**: Element counts and detailed document analysis
- **📋 Table Structure Preservation**: Better context for RAG applications
- **💻 Advanced Code Detection**: Improved language identification
- **📈 Superior Content Extraction**: Better handling of complex documents

### 🎯 Ready for Phase 3:
- **Next Notebook**: `03_embedding_pipeline.ipynb`
- **Enhanced Chunking**: Leverage semantic structure for better chunking
- **Context-Aware Embeddings**: Use element classification for improved RAG

### 📁 Generated Files:
- `data/processed/*_documents.json` - Enhanced document data with semantic elements
- `data/processed/document_metadata.csv` - Rich metadata with element analysis
- `data/processed/parser_comparison.json` - Performance comparison results
- `data/processed/ingestion_summary.json` - Complete pipeline statistics

The enhanced document processing will significantly improve RAG performance through better structure preservation and semantic understanding! 🚀

In [ ]:
# Create a summary report
if pipeline_result["success"]:
    summary_report = {
        "ingestion_date": datetime.now().isoformat(),
        "pipeline_metadata": pipeline_result["pipeline_metadata"],
        "repository_stats": pipeline_result["repository_stats"],
        "parsing_stats": pipeline_result["parsing_stats"],
        "document_stats": pipeline_result["document_stats"],
        "output_files": pipeline_result["output_files"]
    }
    
    # Save summary report
    summary_file = project_root / "data" / "processed" / "ingestion_summary.json"
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary_report, f, indent=2, ensure_ascii=False)
    
    print(f"💾 Saved ingestion summary to: {summary_file.name}")
    
    print(f"\n🎯 Phase 2 Complete - Data Ingestion Pipeline")
    print(f"✅ Successfully processed {pipeline_result['pipeline_metadata']['successful_parses']} documents")
    print(f"📊 Generated {len(pipeline_result['output_files'])} output files")
    print(f"⏱️  Pipeline took: {pipeline_result['pipeline_metadata']['end_time']}")
    
    print(f"\n🚀 Ready for Phase 3: Chunking and Embedding Generation")
    print(f"   Next notebook: 03_embedding_pipeline.ipynb")
    
else:
    print("❌ Cannot create summary due to pipeline failure.")
    print("Please review the errors above and re-run the pipeline.")

## Next Steps

If the data ingestion completed successfully, you're ready to proceed to:
- **Phase 2 TODO 2**: Semantic chunking and embedding generation
- **Notebook 03**: `embedding_pipeline.ipynb`

The processed documents are now saved in the `data/processed/` directory and ready for chunking and embedding generation.

### Key Outputs:
- Document JSON files by type (markdown_documents.json, etc.)
- Comprehensive metadata (document_metadata.csv)
- Pipeline statistics and summary
- Quality assessment and recommendations